In [6]:
!pip install "dlt[duckdb]"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.8/67.8 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 348.9/348.9 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 38.2 MB/s eta 0:00:00


In [18]:
import dlt
from dlt.sources.helpers.rest_client import RESTClient

@dlt.resource(name="rides", write_disposition="replace")
def taxi_rides():
    client = RESTClient(base_url="https://us-central1-dlthub-analytics.cloudfunctions.net")

    page = 1
    prev_first_id = None

    while True:
        # Requesting by page instead of offset
        params = {
            "page": page,
            "limit": 1000
        }
        response = client.get("data_engineering_zoomcamp_api", params=params)
        data = response.json()

        # 1. Stop if empty
        if not data:
            print(f"Stopped: No data on page {page}")
            break

        # 2. Check for duplication (The "Loop Trap")
        current_first_id = data[0].get('Trip_Pickup_DateTime') # Using timestamp as a proxy ID
        if prev_first_id == current_first_id:
            print(f"\n⚠️ Page {page} returned the same data as Page {page-1}.")
            print("The API does not support page-based pagination.")
            break

        print(f"✅ Loaded Page {page} ({len(data)} records)")
        yield data

        prev_first_id = current_first_id
        page += 1

pipeline = dlt.pipeline(pipeline_name="taxi_page_test", destination="duckdb", dataset_name="taxi_data")
pipeline.run(taxi_rides())

✅ Loaded Page 1 (1000 records)
✅ Loaded Page 2 (1000 records)
✅ Loaded Page 3 (1000 records)
✅ Loaded Page 4 (1000 records)
✅ Loaded Page 5 (1000 records)
✅ Loaded Page 6 (1000 records)
✅ Loaded Page 7 (1000 records)
✅ Loaded Page 8 (1000 records)
✅ Loaded Page 9 (1000 records)
✅ Loaded Page 10 (1000 records)
Stopped: No data on page 11


2026-02-28 07:59:26,219|[WARNING]|399|135000070295552|dlt|validate.py|verify_normalized_table:91|In schema `taxi_page_test`: The following columns in table 'rides' did not receive any data during this load and therefore could not have their types inferred:
  - rate_code
  - mta_tax

Unless type hints are provided, these columns will not be materialized in the destination.
One way to provide type hints is to use the 'columns' argument in the '@dlt.resource' decorator.  For example:

@dlt.resource(columns={'rate_code': {'data_type': 'text'}})



LoadInfo(pipeline=<dlt.pipeline(pipeline_name='taxi_page_test', destination='duckdb', dataset_name='taxi_data', default_schema_name='taxi_page_test', schema_names=['taxi_page_test'], first_run=False, dev_mode=False, is_active=True, pipelines_dir='/var/dlt/pipelines', working_dir='/var/dlt/pipelines/taxi_page_test')>, metrics={'1772265543.3443127': [{'started_at': DateTime(2026, 2, 28, 7, 59, 26, 263146, tzinfo=Timezone('UTC')), 'finished_at': DateTime(2026, 2, 28, 7, 59, 28, 355780, tzinfo=Timezone('UTC')), 'job_metrics': {'rides.e274a78e17.insert_values.gz': LoadJobMetrics(job_id='rides.e274a78e17.insert_values.gz', file_path='/var/dlt/pipelines/taxi_page_test/load/normalized/1772265543.3443127/started_jobs/rides.e274a78e17.0.insert_values.gz', table_name='rides', started_at=DateTime(2026, 2, 28, 7, 59, 26, 340217, tzinfo=Timezone('UTC')), finished_at=DateTime(2026, 2, 28, 7, 59, 28, 269652, tzinfo=Timezone('UTC')), state='completed', remote_url=None, retry_count=0), '_dlt_pipeline_st

In [20]:
import duckdb

# Connect to the database created in your last run
conn = duckdb.connect("taxi_page_test.duckdb")

# Execute the final calculation
res = conn.execute("""
    SELECT
        COUNT(*) AS total_records,
        MIN(trip_pickup_date_time) AS start_time,
        MAX(trip_pickup_date_time) AS end_time,
        (COUNT(CASE WHEN payment_type = 'Credit' THEN 1 END) * 100.0 / COUNT(*)) AS credit_card_percentage,
        SUM(tip_amt) AS total_tips
    FROM taxi_data.rides
""").fetchone()

print("="*40)
print("🚕 FINAL HOMEWORK ANSWERS 🚕")
print("="*40)
print(f"Total Records: {res[0]}")
print(f"Q1 Date Range: {res[1]} to {res[2]}")
print(f"Q2 CC %:      {res[3]:.2f}%")
print(f"Q3 Total Tips: ${res[4]:,.2f}")
print("="*40)

🚕 FINAL HOMEWORK ANSWERS 🚕
Total Records: 10000
Q1 Date Range: 2009-06-01 11:33:00+00:00 to 2009-06-30 23:58:00+00:00
Q2 CC %:      26.66%
Q3 Total Tips: $6,063.41
